# 표현 (Representation)

**과제 내용**

1. **TDM 생성** — 뉴스기사 10개를 선택해 전처리(문서별 토큰 저장)하고, 문서별 단어 빈도로 TDM 생성
   - 문서 수가 고정되어 있을 때, TDM의 크기를 줄이기 위한 전처리 방법은?
2. **TF-IDF 계산** — TDM 기반으로 각 단어의 TF-IDF 점수 계산
   - TF-IDF가 TDM보다 유리한 점은? TF와 IDF 각각의 의미는?
3. **N-gram** — 2-gram, 3-gram 생성 후 세 개의 n-gram으로 TDM 생성
   - N-gram의 크기를 선택할 때 고려할 사항은? n이 커지면 어떤 장단점이 있는가?
4. **단어-문맥 동시등장 행렬** — 기사 하나를 골라 window 크기만큼 이동하며 동시 등장 단어 카운트
   - window_size를 설정할 때 고려할 점은? 크거나 작을 때 각각 어떤 영향이 있는가?

---

**분석 대상 데이터**

- 출처 : 지디넷코리아 (네이버 뉴스 IT/과학 섹션 제휴 언론사) 기사 10건
- 선정 기준 : TF-IDF와 문서 유사도가 의미 있게 나오도록 주제를 **5개 군집**으로 분산  
  (AI/보안 3건, 게임 2건, 자동차·배터리 2건, 반도체 1건, 디바이스·플랫폼 2건)

---
## 0. 환경 설정

한국어 형태소 분석을 위해 KoNLPy(Komoran)를 사용

In [1]:
%pip install -q konlpy scikit-learn nltk
%pip install -q gensim

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import json
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)

### 0.1 데이터 : 뉴스 기사 10건

각 기사를 `category / title / url / text` 로 정리

In [17]:
NEWS = [
    {
        "category": "AI/보안",
        "title": "올 2분기 AI·LLM서 발생한 취약점, 전년 比 14배 폭증",
        "url": "https://zdnet.co.kr/view/?no=20260907105713",
        "text": """올해 2분기 기준 인공지능(AI) 및 대규모 언어 모델(LLM)에서 발생한 보안 취약점이 전년 동기 대비 14배가량 늘어난 것으로 나타났다. 글로벌 사이버보안 기업 카스퍼스키는 7일 이같은 내용을 담은 '분기별 익스플로잇 및 취약점 보고서'를 발표했다. 보고서에 따르면 AI 도입이 다양한 산업군으로 빠르게 확대하면서 AI 서비스에서 발견되는 취약점도 크게 늘어난 것으로 나타났다. 올해 2분기 기준 AI 및 LLM 서비스에서 기록된 취약점은 935건으로 2025년 4분기 93건 대비 10배 늘어난 것으로 나타났다. 1년 전 같은 기간과 비교하면 약 14배 늘어난 수치다. 보고서에 따르면 AI 관련 소프트웨어에서 주요 문제로 나타난 것은 시스템 객체에 대한 취약한 접근 제어, 인증 및 권한 부여 구현 오류, 인젝션 취약점인 것으로 나타났다. AI 서비스 자체의 취약점뿐만 아니라 AI 서비스로 위장한 악성 소프트웨어를 이용한 공격도 증가하고 있는 것으로 나타났다. 카스퍼스키의 최근 중소기업 위협 보고서에 따르면, 2026년 첫 4개월 동안 카스퍼스키 보안 솔루션은 인기 AI 서비스로 위장한 PC용 악성 또는 원치 않는 소프트웨어를 이용해 중소기업을 공격한 사례를 3만3300건 이상 탐지했다. 전년 동기 대비 5배가량 급증한 수치다. 이에 AI 기술과 서비스가 기업의 업무 환경에 빠르게 통합됨에 따라 AI 관련 취약점과 이를 악용한 위협에 대한 지속적인 관리의 중요성도 커지고 있다는 분석이 나온다. 알렉산더 콜레니스코프 카스퍼스키 멀웨어 전문가는 "AI 기술은 업무를 효과적으로 자동화하는 데 도움을 주지만, 조직 전반에서 AI가 빠르게 도입되면서 새로운 보안 과제가 발생하고 있다"며 "접근 제어와 인증, 인젝션 처리 과정의 취약점이 가장 빈번하게 나타나는 유형으로 확인되고 있다"고 분석했다.""",
    },
    {
        "category": "AI",
        "title": "AI '데이터 벽'에 갇힌다...SKT, 사후학습·추론으로 돌파구",
        "url": "https://zdnet.co.kr/view/?no=20260907100119",
        "text": """AI 모델 성능을 높이는 데 필요한 학습 데이터가 부족해지는 '데이터 벽' 문제가 현실이 되고 있다. 단순히 공개된 데이터의 고갈에 그치지 않고 웹 접근 제한과 합성 데이터의 품질 문제까지 겹치면서 AI 경쟁 방식도 달라질 것이란 분석이다. 7일 SK텔레콤 뉴스룸에 따르면 회사 AI정책연구원은 기고를 통해 AI 업계가 맞닥뜨린 데이터 제약을 공개 텍스트의 물량, 웹 데이터 접근성, 합성 데이터 품질 등 세 가지로 구분했다. 먼저 데이터 총량에는 한계가 있다. 미국 AI 연구기관 에포크AI는 인간이 작성한 고품질 공개 텍스트를 약 300조 토큰으로 추산했다. 현재와 같은 AI 확장 추세가 이어지면 2032년 이내에 소진될 가능성이 있다. 학습 방식과 데이터 활용 효율에 따라 시점은 크게 달라질 것으로 봤다. 중요한 것은 고갈 시점보다 데이터의 한계효용이다. 데이터를 계속 늘리는 것만으로 얻을 수 있는 성능 향상에는 한계가 나타나고 있어 앞으로는 같은 데이터에서 얼마나 높은 학습 효율을 끌어내느냐가 중요해진다는 설명이다. 웹에 있는 데이터를 자유롭게 가져다 쓰기도 어려워지고 있다. 클라우드플레어는 오는 15일부터 광고가 게재된 페이지에서 AI 학습 에이전트용 크롤러를 기본 차단하고 검색과 학습을 함께 수행하는 크롤러에 대한 관리도 강화한다. 데이터가 웹에 존재하는 것과 AI 기업이 실제 사용할 수 있는 것은 별개의 문제가 된 셈이다. AI가 만든 합성 데이터도 완전한 대안은 아니다. 네이처에 게재된 연구에서는 모델이 생성한 데이터로 후속 모델을 반복 학습하면 희귀 사례와 예외적 패턴이 사라지면서 성능이 퇴화하는 모델 붕괴 가능성이 제기됐다. 수학이나 코딩처럼 결과 검증이 쉬운 분야에서는 유용하지만 인간이 만든 데이터를 전면적으로 대체하기는 어렵다는 분석이다. 이 같은 변화로 데이터의 경제적 가치도 달라지고 있다. 오픈AI는 악셀 슈프링어와 뉴스코프 등 미디어 기업과 계약을 맺고 콘텐츠 이용 접근 권한을 확보하고 있다. 레딧도 실시간 데이터 API를 AI 기업에 제공한다. 데이터를 한 번 판매하는 데 그치지 않고 지속적인 접근권을 제공하는 라이선싱 방식이 중요해지고 있다. AI를 활용하는 일반 기업에는 공개 데이터 고갈보다 내부 데이터를 얼마나 제대로 활용할 수 있느냐가 더 중요한 문제다. 거래 운영 로그와 고객 상담 기록, 설계 정비 이력, 현장 계측치 등 외부에서 구하기 어려운 데이터의 상대적 가치가 높아지기 때문이다. 데이터를 많이 보유했다고 곧바로 AI 경쟁력이 생기는 것은 아니다. 여러 시스템에 흩어진 데이터를 정제하고 활용 목적과 권리관계를 명확히 한 뒤 실제 업무와 연결할 수 있어야 한다. 처음부터 모델 학습에 투입하기보다 검색증강생성으로 데이터를 활용하고 효과가 확인된 분야부터 사후학습을 확대하는 방식도 대안으로 제시됐다. 그런 가운데 SK텔레콤은 모델, 데이터, 인프라 측면에서 대응하고 있다. 자체 AI 모델 개발에서는 상대적으로 부족한 고품질 한국어 데이터를 발굴하고 정제하는 데 집중하고, 보유 데이터는 개인정보 보호와 비식별화 등을 전제로 활용한다.""",
    },
    {
        "category": "AI",
        "title": "오픈AI '아스트라', 보안·안전 기능도 막강...7대 기능 살펴보니",
        "url": "https://zdnet.co.kr/view/?no=20260906142425",
        "text": """미국 오픈AI가 3일 자사 최신 인공지능(AI) 모델 'GPT-6 아스트라'를 공개했다. 이 모델에 대해 오픈AI는 범용인공지능(AGI)으로 간주될 수 있다고 밝혀 시선을 모았다. 실제 오픈AI 그렉 브록먼 사장은 제품 브리핑에서 "세계 최고 컴퓨터 사용자 모델"이라면서 "이 새로운 모델이 언젠가는 인공 일반 지능의 초기 단계로 여겨질 것"이라고 밝혔다. AGI는 챗GPT처럼 특정 작업을 잘하는게 아니라 사람처럼 다양한 분야를 이해하고 배우며 문제를 해결할 수 있는 AI를 말한다. 아스트라는 사이버보안과 안전 측면에서도 주목할만 하다. 아스트라의 보안 능력에 대해 오픈AI는 "우리의 준비성 평가 프레임워크에서 사이버보안 역량이 처음으로 치명적 수준에 도달한 모델"이라고 밝혔다. 준비성 평가 프레임워크는 오픈AI가 AI 모델의 능력이 어느 수준까지 올라갔을 때 심각한 위험이 발생할 수 있는지를 사전에 측정하고, 그 수준에 맞춰 안전장치를 강화하기 위해 만든 자체 위험관리 기준이다. 2023년 12월 처음 공개했다. 이후 모델 능력이 높아지면서 계속 개정하고 있다. 실제 오픈AI는 최근 몇 주간 아스트라 출시를 늦추며 사이버 악용과 승인되지 않은 모델 행동을 막기 위한 안전장치를 강화했다. 오픈AI는 아스트라가 사이버 역량에서 상당한 수준의 발전을 이뤘다고 강조했다. 회사가 설정한 치명적 임계치를 충족했다는 것이다. 이는 아스트라가 적절한 도구와 접근 권한을 갖게 될 경우, 이전에 알려지지 않았던 보안 취약점을 찾아내고 새로운 공격과 악용 방법을 개발할 수 있으며, 보안이 잘 갖춰진 다양한 시스템에서도 사람이 매 단계마다 직접 지시하지 않더라도 이러한 작업을 수행할 수 있다는 것을 의미한다고 오픈AI는 설명했다. 오픈AI는 "이에 따라 우리는 모델이 유해한 사이버 행위를 수행하는 것을 막기 위한 보호장치를 대폭 강화했다"고 말했다. 실제 오픈AI는 더 엄격한 격리와 체크포인트 암호화, 사고 과정을 포함한 전체 실행 경로에 대한 전면적인 모니터링을 실시했다. 또 모델을 내부적으로 사용하기 전에 차단형 정렬 평가 절차를 거치게 했다. AI의 견고성은 예상하지 못한 입력이나 작은 변화가 생겨도 성능과 판단을 안정적으로 유지하는 능력을 말한다. 오픈AI는 아스트라가 새로운 견고성 안전 훈련 기법을 적용해 이전 모델보다 탈옥 공격에 훨씬 더 강한 견고성을 갖게 됐다고 밝혔다. 오픈AI는 아스트라가 모델 정렬 측면에서도 중요한 진전을 이뤘다고 밝혔다. 여기에는 사전학습 데이터 구성 방식부터 강화학습 과정 평가에 이르기까지 정렬을 개선하기 위한 다양한 조치가 포함됐다.""",
    },
    {
        "category": "게임",
        "title": "라이엇게임즈 공동 창업자 \"LoL MMO는 '성배'…개발 지속, 미래 밝다\"",
        "url": "https://zdnet.co.kr/view/?no=20260907101502",
        "text": """라이엇 게임즈가 '리그 오브 레전드' 세계관 기반 MMORPG 개발을 지속하고 있다고 외신 IGN이 6일 보도했다. 보도에 따르면 마크 메릴 라이엇 게임즈 공동 창업자는 최근 PAX 행사에서 킨다 퍼니 게임즈의 그렉 밀러와 만나 LoL MMO가 여전히 개발 중이라고 밝혔다. 메릴은 사내에서 이 프로젝트를 '성배 탐색'이라 부른다며 "수많은 기사가 성배를 찾다 쓰러지겠지만, 성배이기에 도전하는 것"이라고 비유했다. 이어 "이용자들의 기대가 무엇인지 알고 있다"며 "대규모 조직에서 이를 유지하기란 쉽지 않지만 잘 진행하고 있으며, 미래는 대단히 밝다"고 덧붙였다. 외신 측은 라이엇 게임즈가 2021년 8월 해당 MMO 개발을 공식화했으나, 주요 개발진 이탈과 프로젝트 리셋이 겹치며 난항을 겪었다고 보도하기도 했다. 2022년 그렉 스트리트 당시 총괄 프로듀서가 "출시를 장담할 수 없다"고 밝힌 뒤 이듬해 퇴사했고, 2024년에는 개발 방향성을 원점에서 재검토했다고 알려졌다.""",
    },
    {
        "category": "게임",
        "title": "스마일게이트 신작 '이클립스', MMORPG 공식 비틀었다",
        "url": "https://zdnet.co.kr/view/?no=20260906052156",
        "text": """스마일게이트가 오는 10일 정식 출시를 앞둔 신작 MMORPG '이클립스: 더 어웨이크닝'의 핵심 게임성과 서비스 계획을 공개했다. 스마일게이트는 지난 5일 공식 유튜브 채널을 통해 '이클립스 론칭 프리뷰' 라이브 방송을 진행하고, 신규 트레일러와 함께 인게임 핵심 시스템, 보스전, PvP 전장 및 향후 업데이트 로드맵을 상세히 소개했다. 이날 방송에는 개발사 엔픽셀의 이상문 총괄 PD를 비롯해 방송인 허준, CM 박가영, 스트리머 빅보스와 수삼이 참석해 이용자들과 소통에 나섰다. 사전 예약자 수 100만명을 돌파하며 주목받고 있는 이클립스는 오는 10일 낮 12시 정식 론칭한다. 사전 다운로드는 이달 7일 공식 커뮤니티 안내를 통해 진행될 예정이다. 이날 방송에서 이상문 PD는 이용자의 시간 효율을 배려한 핵심 시스템으로 '이클립스 타임'을 소개했다. 이클립스 타임은 이용자가 원하는 순간에 직접 켜고 끌 수 있는 성장 버프 시스템이다. 특히 전장 등 부가 콘텐츠에 진입할 경우 버프가 자동으로 비활성화 처리되어 시간적 손실을 방지하도록 설계됐다. 캐릭터 성장의 중추 역할을 맡는 '권능' 시스템도 베일을 벗었다. 권능은 성흔 6종과 성유물 3종의 조합으로 활성화된다. 몬스터 처치나 NPC 상인을 통해 획득 가능한 성흔은 거래가 가능하나 감정 후에는 귀속으로 전환된다. 성소와 분해 재료인 권능 조각으로 제작하는 성유물은 거래가 불가능하다. 이용자는 동일 등급의 성흔 6개와 성유물을 장착해 세트를 완성하고, 다양한 권능 스킬을 전략적으로 선택해 활용할 수 있다. 장비 제작 및 파밍 구조에 대한 설명도 이어졌다. 일반 장비 외에 세트 및 유니크 장비가 존재하며, 세트 장비는 솔라리온 결정, 엘룬 결정 등 핵심 재료를 통해 제작할 수 있다. 편의 기능으로는 미니맵 우측 보스 스케줄 아이콘을 통해 등장 시간 확인 및 월드맵 화면 포커싱 기능을 지원해 파밍 편의성을 높였다. 보스 레이드와 경쟁 콘텐츠도 공개됐다. 대권역별로 1개체씩 등장하는 어비스 보스는 등장 시 해당 지역이 일시적으로 PvP 가능 구역으로 변경되는 특징을 지녔다. 이와 함께 이용자를 삼켜 체내에서 탈출해야 하는 기믹을 지닌 샤를레, 시야를 차단하는 장막 패턴의 아르칸다르, 다수의 이용자가 협력하는 월드보스 크라카와 에버하르트 등 다채로운 공략 요소를 선보였다. 보스 처치 시 거래 가능한 고가치 장비와 성장 재료가 드롭되며, 공격에 참여한 이용자 전원에게 별도의 참여 보상 상자가 지급된다. PvP 콘텐츠로는 10대10 진영전 하이리버 전장이 소개됐다. 고저차가 존재하는 지형에서 고지대 시야 제한 및 역습의 위험 등 입체적인 전략 요소가 가미됐으며, 연속 킬을 달성한 상대를 제압할 경우 대량 득점이 가능한 연속 처치 스코어 시스템을 적용해 역전 가능성을 열어뒀다. 서버 간 대결을 지원하는 인터서버 던전은 일반 던전 보스층과 주간 던전으로 나뉘어 운영된다. 매칭 그룹은 특정 주기마다 재조정되어 특정 세력 간의 구도 고착화를 방지할 방침이다.""",
    },
    {
        "category": "자동차/배터리",
        "title": "中, 1~7월 전기차 배터리 점유율 70% 넘겨",
        "url": "https://zdnet.co.kr/view/?no=20260907102324",
        "text": """올해 1월부터 7월까지 전세계 전기차 배터리 시장을 분석한 결과, 점유율 상위 10곳 중 중국 기업들의 점유율이 72.8%를 기록한 것으로 조사됐다. 7일 시장조사업체 SNE리서치는 이같은 조사 결과를 내놨다. SNE리서치에 따르면 이 기간 세계 각국에 등록된 전기차, 플러그인하이브리드차, 하이브리드차에 탑재된 총 배터리 사용량은 약 725.2GWh로 전년 동기 대비 20.4% 성장했다. 7월 한 달 사용량은 116GWh로 전년 동월 대비 22.1% 증가해, 월간 기준으로도 누적 성장률을 웃도는 흐름이 이어졌다. CATL은 전년 동기 대비 26.6% 증가한 289.6GWh로 1위를 기록했다. 점유율은 38%에서 39.9%로 상승해 40% 선에 근접했다. 2위 BYD는 106.7GWh로 4.7% 증가하는 데 그쳤고, 점유율은 16.9%에서 14.7%로 하락했다. 두 업체의 합산 점유율은 54.6%로 전년 동기 54.9% 대비 낮아졌다. 국내 업체 가운데 LG에너지솔루션은 전년 동기 대비 4.5% 증가한 60.3GWh를 기록하며 3위를 유지했다. 테슬라, 현대차그룹, GM, 폭스바겐 등 주요 글로벌 완성차에 대한 공급이 이어지며 사용량은 늘었지만 점유율은 9.6%에서 8.3%로 하락했다. SK온은 22.3GWh로 전년 동기 대비 9.8% 감소하며 8위를 기록했고, 점유율은 4.1%에서 3.1%로 낮아졌다. SK온 배터리는 현대차그룹, 포드, 폭스바겐, 메르세데스벤츠 등에 탑재되고 있으나, 북미와 유럽 일부 고객사의 전기차 생산 계획 조정이 이어지면서 사용량 회복이 지연되고 있다. 일본 파나소닉은 전년 동기 대비 7.6% 증가한 26.2GWh로 6위에 자리했다. 주요 고객사인 테슬라의 북미 판매가 사용량 증가를 뒷받침했지만 시장 성장률에는 미치지 못하면서 점유율은 4%에서 3.6%로 하락했다. 중국계 업체들은 상위권 전반에서 30%를 넘는 성장률을 이어갔다. CALB는 전년 동기 대비 34.3% 증가한 37.3GWh로 4위, 고션은 44.2% 늘어난 34GWh로 5위를 기록했다. EVE는 53.1% 성장한 25GWh로 7위, 에스볼트는 39.1% 증가한 18.9GWh로 9위에 올랐다. 특히 REPT는 118.5% 증가한 16.9GWh를 기록하며 상위 10개사에 새로 진입했다. 직전까지 10위였던 신왕다는 근소한 차이로 순위권 밖으로 밀렸다. 이들 5개사에 CATL과 BYD를 더한 중국계 7개사의 합산 점유율은 72.8%로 전년 동기 대비 상승했다. 내수 기반의 물량과 LFP 중심의 원가 경쟁력을 바탕으로 규모를 키우는 한편, 해외 완성차와 상용차 및 에너지저장장치 등으로 공급 영역을 넓힌 결과로 풀이된다. 지역별로는 유럽이 29.3%, 중국이 16.6% 늘고 아시아가 77.1%, 남미가 192.6% 급증한 반면, 북미는 20.2% 감소했다. 미국은 연방 전기차 세액공제가 지난해 9월 말 종료된 뒤 순수 전기차의 신차 판매 비중이 6%대로 낮아졌고, 포드 라이트닝과 혼다 프롤로그, 폭스바겐 등 주요 차종이 잇따라 단종되며 배터리 수요가 직접적인 타격을 받았다. 유럽은 상반기 순수 전기차 등록이 35% 늘며 반기 기준 처음으로 가솔린차 비중을 넘어섰다.""",
    },
    {
        "category": "자동차",
        "title": "BMW, 전기차 누적 인도 200만대 돌파…i3 첫 인도 이후 13년만",
        "url": "https://zdnet.co.kr/view/?no=20260907101115",
        "text": """BMW그룹이 순수전기차 누적 인도 200만대를 돌파했다. 2013년 말 BMW i3를 처음 고객에게 인도한 이후 약 13년 만이다. BMW그룹코리아는 7일 BMW 그룹이 지난 8월 독일 딩골핑 공장에서 생산한 BMW i5 M60 xDrive를 스페인 고객에게 인도하며 순수전기차 누적 200만대를 기록했다고 밝혔다. 플러그인하이브리드를 포함한 전동화 모델 누적 인도는 약 350만대다. BMW그룹에 따르면 올해 상반기 전 세계 판매 차량 가운데 전동화 모델 비중은 25%를 넘었다. 유럽에서는 순수전기차 비중이 28%까지 높아졌다. 올해 2분기 유럽 내 순수전기차 판매량은 더 뉴 BMW iX3 공급 확대에 힘입어 전년 동기보다 38% 증가한 8만1445대를 기록했다. 국내에서도 BMW의 전기차 판매는 증가세를 보였다. BMW코리아는 올해 1월부터 8월까지 순수전기차 5895대를 판매해 수입 프리미엄 브랜드 가운데 1위를 기록했다고 밝혔다. 전년 동기보다 약 55% 늘어난 수치다. 같은 기간 미니는 국내에서 순수전기차 1576대를 판매했다. 전년 동기 대비 약 55% 증가했으며, 미니 전체 판매량에서 순수전기차가 차지하는 비중은 29% 이상이었다. BMW의 플러그인하이브리드와 순수전기차를 합한 전동화 모델 판매량은 9203대로 집계됐다. 지난해 같은 기간 7736대와 비교해 약 19% 증가했다. BMW그룹코리아는 충전 및 정비 인프라도 확대하고 있다. 2023년 발표한 차징 넥스트 프로젝트를 통해 현재까지 전국에 충전기 3155기를 설치했으며, 연말까지 4000기로 늘릴 계획이다. 지난 6월에는 서울역 인근 BMW 차징 허브 라운지와 인천 영종도 BMW 드라이빙 센터에 공용 400킬로와트 초급속 충전기 4기를 설치했다.""",
    },
    {
        "category": "반도체",
        "title": "마이크론, YMTC 특허 7번째 무효 판단 받았다...쟁점특허 27건",
        "url": "https://zdnet.co.kr/view/?no=20260906193737",
        "text": """미국 마이크론이 중국 YMTC를 상대로 자국에서 힘겨운 특허분쟁을 이어가고 있다. YMTC는 미국에서 마이크론을 상대로 메모리 반도체 특허침해소송 3건을 제기했다. 마이크론은 무효심판 등으로 대응 중이지만 결과는 신통치 않다. YMTC가 소송 3건에 사용한 특허는 모두 27건이다. 마이크론은 27건 중 19건에 대해 특허심판원에 무효심판을 청구했고, 나머지 8건에 대해선 특허청에 결정계 재심사를 신청했다. 6일 업계에 따르면 미국 특허심판원은 지난 2일 YMTC 특허 1건이 무효라고 판단했다. 이로써 마이크론이 특허심판원에 무효심판을 청구한 YMTC 특허 19건 중 무효 판단을 받은 특허는 7건으로 늘었다. 무효 판단에 대해 YMTC는 항소할 수 있다. 마이크론이 YMTC 특허 7건에 대해 무효 판단을 받았지만, 마이크론이 특허심판원에 무효심판을 청구한 YMTC 특허 19건 중 12건은 유효 취지 판단이 나왔다. 유효 취지 판단은 무효심판이 개시되지 않았거나, 심판 개시 후 유효 또는 일부 유효 판단이 나온 특허를 말한다. 특허심판원은 무효심판 심리에서 특허가 무효화될 가능성이 낮다고 판단하면 무효심판을 개시하지 않는다. 특허심판원의 무효심판 개시 기각 결정에 대해선 항소할 수 없다. 마이크론은 특허심판원 심리 후 유효 또는 일부 유효 판단이 나온 YMTC 특허에 대해선 연방항소법원에 항소했다. 마찬가지로, 무효 또는 일부 무효 판단이 나온 특허에 대해선 YMTC가 항소하고 있다. YMTC가 마이크론을 상대로 소송 3건을 제기하며 사용한 특허 27건 중 나머지 8건에 대해 마이크론은 특허청에 결정계 재심사를 청구했다. 8건 중 1건은 기각됐고, 7건은 재심사 명령을 받았다. 마이크론이 결정계 재심사 8건을 청구한 시점은 올해 3월부터 7월이다. 지난해부터 미국 특허청이 특허 안정성 제고를 명분으로 특허심판원의 무효심판 재량적 기각을 늘리면서, 무효심판 개시율이 크게 떨어졌다. 이 때문에 최근 특허청 결정계 재심사를 활용하는 사례가 늘었다. 특허심판원 무효심판이 특허청 결정계 재심사보다 특허권자를 압박하는 힘이 더 크다. YMTC는 지난 2022년 12월 미국 상무부 블랙리스트에 포함된 다음해인 2023년부터 미국에서 마이크론을 상대로 특허소송을 제기했다. YMTC가 미국에서 사업을 하기 어렵기 때문에 미국에서 YMTC는 사실상 특허관리전문기업 같은 지위로서 분쟁을 지속할 수 있다.""",
    },
    {
        "category": "디바이스",
        "title": "레노버, 개인용 AI '키라' 작동 PC·모바일 기기 확대",
        "url": "https://zdnet.co.kr/view/?no=20260907103847",
        "text": """레노버가 7일 개인용 AI 에이전트 '키라' 작동 플랫폼과 연동 앱 및 서비스 확대로 개인화 AI 경험을 강화한다고 밝혔다. 키라는 레노버와 모토로라가 공동 개발한 AI 에이전트 플랫폼으로 올 1월 CES 2026에서 공개됐다. PC, 스마트폰, 태블릿, 웨어러블 등 여러 레노버 기기를 넘나들며 작동하는 것이 특징이다. 키라는 온디바이스 처리를 우선하며 클라우드 연동이 필요한 경우에도 요청 처리에 필수적인 최소한의 프롬프트와 맥락 정보만 보안 암호화 채널을 거쳐 전송된다. 이용자가 지식 기반에 추가한 관련 정보도 암호화해 기기 안에만 저장한다. 레노버는 키라의 기반이 되는 로컬 AI 모델 최적화를 거쳐 메모리 16GB인 씽크패드와 아이디어패드, 요가, 리전 등 레노버 PC 제품으로 지원을 확대했다. 해당 기기 이용자들은 레노버 공식 웹사이트에서 키라를 다운로드하고 설치해 활용 가능하며 향후 출시될 지원 기기에는 키라가 사전 탑재된다. PC 제품 이외에 안드로이드17 업그레이드가 가능한 일부 모토로라 기기, 모토워치 울트라 등 웨어러블 기기도 키라를 지원할 예정이다. 현재 모토AI를 탑재한 스마트폰, AI 나우를 탑재한 태블릿은 안드로이드17 업데이트 이후 각각 이를 키라로 대체한다.""",
    },
    {
        "category": "플랫폼",
        "title": "우버, 나이지리아서 돌연 철수…경쟁당국 조사 나서",
        "url": "https://zdnet.co.kr/view/?no=20260907085716",
        "text": """나이지리아 경쟁당국이 차량 호출 플랫폼 우버의 갑작스러운 현지 사업 철수와 관련해 조사에 착수했다. 우버가 이용자들에게 별다른 사전 고지 없이 서비스를 중단하면서 소비자 피해가 발생했는지를 들여다보겠다는 취지다. 나이지리아 연방경쟁소비자보호위원회는 우버의 철수 과정과 관련해 조사를 진행하고 있으며, 특히 고객에게 제공하기로 한 서비스가 제대로 이행되지 않았는지를 확인하고 있다고 밝혔다. 툰지 벨로 부위원장은 이에 대해 "우버의 철수 방식과 관련해 조사하고 있으며, 특히 고객들에게 제공되지 못한 서비스가 있는지를 살펴보고 있다"고 입장을 밝혔다. 외신에 따르면 우버는 지난 2일 나이지리아와 우간다에서 서비스를 중단했다. 이번 조치는 우버가 전 세계 사업을 재검토하는 과정에서 이뤄졌으며, 회사는 동시에 전체 인력의 약 10%에 해당하는 3300명을 감원했다. 문제는 서비스 종료 사실이 나이지리아 이용자들에게 사전에 제대로 안내되지 않았다는 점이다. 이로 인해 차량을 이용하려던 승객과 플랫폼을 통해 영업하던 운전자들 사이에서 혼란이 발생했다. 우버 측 대변인은 외신의 질문에 철수에 대한 구체적인 이유와 입장을 밝히지 않았다.""",
    },
]

for d in NEWS:
    print(f"[{d['category']}]  {len(d['text'])}자  {d['title'][:40]}")

[AI/보안]  900자  올 2분기 AI·LLM서 발생한 취약점, 전년 比 14배 폭증
[AI]  1515자  AI '데이터 벽'에 갇힌다...SKT, 사후학습·추론으로 돌파구
[AI]  1281자  오픈AI '아스트라', 보안·안전 기능도 막강...7대 기능 살펴보니
[게임]  493자  라이엇게임즈 공동 창업자 "LoL MMO는 '성배'…개발 지속, 미래 밝
[게임]  1474자  스마일게이트 신작 '이클립스', MMORPG 공식 비틀었다
[자동차/배터리]  1561자  中, 1~7월 전기차 배터리 점유율 70% 넘겨
[자동차]  859자  BMW, 전기차 누적 인도 200만대 돌파…i3 첫 인도 이후 13년만
[반도체]  1183자  마이크론, YMTC 특허 7번째 무효 판단 받았다...쟁점특허 27건
[디바이스]  614자  레노버, 개인용 AI '키라' 작동 PC·모바일 기기 확대
[플랫폼]  578자  우버, 나이지리아서 돌연 철수…경쟁당국 조사 나서


### 0.2 전처리 파이프라인

과제1에서 만든 파이프라인을 함수로 재사용한다.

`정제 → 형태소 분석(Komoran + 사용자사전) → 품사 필터링 → 표제어 복원 → 불용어 제거`

In [19]:
from konlpy.tag import Komoran

# --- 사용자 사전 : 분석기가 쪼개버리는 고유명사/전문용어를 등록 ---
USER_DIC = """\\
카스퍼스키\tNNP
오픈AI\tNNP
아스트라\tNNP
클라우드플레어\tNNP
에포크AI\tNNP
스마일게이트\tNNP
라이엇\tNNP
이클립스\tNNP
엔픽셀\tNNP
SNE리서치\tNNP
LG에너지솔루션\tNNP
SK온\tNNP
SK텔레콤\tNNP
마이크론\tNNP
YMTC\tNNP
특허심판원\tNNP
레노버\tNNP
모토로라\tNNP
나이지리아\tNNP
우버\tNNP
파나소닉\tNNP
폭스바겐\tNNP
현대차그룹\tNNP
개인용\tNNG
전기차\tNNG
순수전기차\tNNG
배터리\tNNG
취약점\tNNG
데이터\tNNG
점유율\tNNG
"""

DIC_PATH = os.path.join(OUT_DIR, "user_dic.txt")
with open(DIC_PATH, "w", encoding="utf-8") as f:
    f.write(USER_DIC)

komoran = Komoran(userdic=DIC_PATH)
print(komoran.pos("마이크론이 YMTC 특허심판원 무효심판에서 이겼다."))

[('마이크론', 'NNP'), ('이', 'JKS'), ('YMTC', 'NNP'), ('특허심판원', 'NNP'), ('무효', 'NNP'), ('심판', 'NNP'), ('에서', 'JKB'), ('이기', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]


In [20]:
# 분석에 남길 품사만 지정
KEEP_POS  = {"NNG", "NNP", "VV", "VA", "MAG", "SL"}
PREDICATE = {"VV", "VA"}  # 어간 + '다' 로 표제어 복원

STOP_WORDS = set("""
것 등 수 때 중 점 내 외 위 뒤 앞 안 밖 곳 만큼 가지 대비 기준 경우 관련 통해 대해 위해
하다 되다 있다 없다 이다 아니다 같다 보다 두다 오다 가다 내다 지다 늘다 낮다 높다 크다 많다 적다
위하다 대하다 통하다 따르다 나타나다 이어지다 밝히다 전하다 설명하다 강조하다 기록하다
이번 지난 최근 올해 내년 현재 향후 이후 이전 지난해 전년 동기 기간
우리 그것 이것 이런 그런 이어 다만 특히 또한 함께 가장 매우 이미 실제 대한 각각
기자 사진 제공 자료 출처 보도 외신 업계 회사 밝혀 소개 공개 발표 진행 확대 강화
""".split())

KEEP_ONE_CHAR = {"칩", "팀", "랙", "차", "웹", "앱", "폰", "법"}


def clean_text(text):
    """분석에 방해되는 요소 제거 (기자 이메일, URL, 말머리, 특수문자)"""
    text = re.sub(r"\([^)]*@[^)]*\)", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(r"[^가-힣A-Za-z0-9\s\.\,\?\!\'\"%\-\(\)]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess(text):
    """문자열 -> 분석용 토큰 리스트"""
    tokens = []
    for word, pos in komoran.pos(clean_text(text)):
        if pos not in KEEP_POS:
            continue
        if pos in PREDICATE:
            word = word + "다"
        elif pos == "SL":
            word = word.upper() 
        if word in STOP_WORDS:
            continue
        if len(word) == 1 and word not in KEEP_ONE_CHAR:
            continue
        # Komoran 사전에는 '라이엇 게임즈'처럼 공백을 포함한 고유명사가 있다.
        # 한 단어로써 처리할 때에는 공백 대신 '_' 로 바꿔둔다.
        tokens.append(word.replace(" ", "_"))
    return tokens


print(preprocess("구글이 대만과 협력하며 새로운 AI 칩을 설계하고 있다."))

['구글', '대만', '협력', '새롭다', 'AI', '칩', '설계']


---
## 1. TDM (Term-Document Matrix) 생성

### 1.1 문서별 전처리 및 토큰 저장

In [22]:
doc_ids    = [f'D{str(i).zfill(2)}' for i in range(len(NEWS))]
doc_titles = [d["title"] for d in NEWS]
doc_cats   = [d["category"] for d in NEWS]
doc_ls     = [preprocess(d["text"]) for d in NEWS]      # ★ 문서별 토큰 리스트

for i, tokens in enumerate(doc_ls):
    print(f"{doc_ids[i]} [{doc_cats[i]:>8}] 토큰 {len(tokens):>4}개 / 고유 {len(set(tokens)):>3}개"
          f" | {doc_titles[i][:30]}")
    print(f"        {tokens[:12]}")

D00 [   AI/보안] 토큰  135개 / 고유  82개 | 올 2분기 AI·LLM서 발생한 취약점, 전년 比 14
        ['분기', '인공지능', 'AI', '규모', '언어', '모델', 'LLM', '발생', '보안_취약점', '늘어나다', '글로벌', '사이버']
D01 [      AI] 토큰  277개 / 고유 170개 | AI '데이터 벽'에 갇힌다...SKT, 사후학습·추론
        ['AI', '모델', '성능', '높이다', '필요', '학습', '데이터', '부족', '데이터', '문제', '현실', '단순히']
D02 [      AI] 토큰  217개 / 고유 141개 | 오픈AI '아스트라', 보안·안전 기능도 막강...7대
        ['미국', '오픈AI', '자사', '최신', '인공지능', 'AI', '모델', 'GPT', '아스트라', '모델', '오픈AI', '범용']
D03 [      게임] 토큰   71개 / 고유  63개 | 라이엇게임즈 공동 창업자 "LoL MMO는 '성배'…개
        ['라이엇_게임즈', '리그_오브_레전드', '세계관', '기반', 'MMORPG', '개발', '지속', 'IGN', '마크', '메릴', '라이엇_게임즈', '공동']
D04 [      게임] 토큰  296개 / 고유 207개 | 스마일게이트 신작 '이클립스', MMORPG 공식 비틀
        ['스마일게이트', '정식', '출시', '앞두다', '신작', 'MMORPG', '이클립스', '핵심', '게임', '서비스', '계획', '스마일게이트']
D05 [ 자동차/배터리] 토큰  229개 / 고유 136개 | 中, 1~7월 전기차 배터리 점유율 70% 넘겨
        ['1월', '7월', '세계', '전기차', '배터리', '시장', '분석', '결과', '점유율', '상위', '중국', '기업']
D06 [     자동차] 토큰  136개 / 고유  81개 | BMW, 전기차 누적 인도

In [26]:
# 문서별 처리된 토큰을 파일로 저장
with open(f'{OUT_DIR}/doc_tokens.json', "w") as f:
    json.dump(
        [
            {"id": doc_ids[i], "title": doc_titles[i], "category": doc_cats[i], "tokens": doc_ls[i]}
            for i in range(len(NEWS))],
        f, ensure_ascii=False, indent=2)
print(f"-> {OUT_DIR}/doc_tokens.json 저장 완료")

-> output/doc_tokens.json 저장 완료


### 1.2 직접 구현

`{단어 : 인덱스}` 사전을 만들고 (단어 x 문서) 행렬에 빈도를 채워 넣는다.

In [ ]:
word2id = defaultdict(lambda: len(word2id))
[word2id[token] for doc in doc_ls for token in doc]      # 등장 순서대로 인덱스 부여

TDM = np.zeros((len(word2id), len(doc_ls)), dtype=int)   # (단어 x 문서)
for j, doc in enumerate(doc_ls):
    for token in doc:
        TDM[word2id[token], j] += 1

sorted_vocab = sorted((idx, w) for w, idx in word2id.items())
vocab = [w for _, w in sorted_vocab]

df_TDM = pd.DataFrame(TDM, index=vocab, columns=doc_ids)
df_TDM.index.name = "단어"

print("TDM shape (단어 x 문서) :", df_TDM.shape)
df_TDM.loc[df_TDM.sum(axis=1).sort_values(ascending=False).index[:15]]

TDM shape (단어 x 문서) : (846, 10)


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
단어,,,,,,,,,,
AI,12,13,4,0,0,0,0,0,6,0
데이터,0,29,1,0,0,0,0,0,0,0
모델,1,7,12,0,0,0,3,0,1,0
무효,0,0,0,0,0,0,0,18,0,0
특허,0,0,0,0,0,0,0,16,0,0
YMTC,0,0,0,0,0,0,0,14,0,0
서비스,6,0,0,0,1,0,0,0,1,5
증가,1,0,0,0,0,9,3,0,0,0
오픈AI,0,1,12,0,0,0,0,0,0,0


## 1.3 sklearn CountVectorizer 활용

문서는 토큰화 진행한 것을 공백으로 이어붙여 전달한다.

In [29]:
from sklearn.feature_extraction.text import CountVectorizer

docs_joined = [" ".join(tokens) for tokens in doc_ls]

count_vect = CountVectorizer(lowercase=False) # AI -> ai 금지
DTM = count_vect.fit_transform(docs_joined)              # (문서 x 단어)

sk_vocab = count_vect.get_feature_names_out()
df_TDM_sk = pd.DataFrame(DTM.toarray().T, index=sk_vocab, columns=doc_ids)
df_TDM_sk.index.name = "단어"

print("DTM shape (문서 x 단어) :", DTM.shape)
print("직접 구현 어휘 수 :", len(vocab), " / sklearn 어휘 수 :", len(sk_vocab))

check = ["AI", "데이터", "배터리", "특허", "이용자"]
df_TDM_sk.loc[check]

DTM shape (문서 x 단어) : (10, 844)
직접 구현 어휘 수 : 846  / sklearn 어휘 수 : 844


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
단어,,,,,,,,,,
AI,12,13,4,0,0,0,0,0,6,0
데이터,0,29,1,0,0,0,0,0,0,0
배터리,0,0,0,0,0,4,0,0,0,0
특허,0,0,0,0,0,0,0,16,0,0
이용자,0,0,0,1,7,0,0,0,2,2


## 1.4 TDM 의 크기와 희소성(sparsity)

In [32]:
n_terms, n_docs = TDM.shape
nnz = int(np.count_nonzero(TDM))
print(f"행렬 크기       : {n_terms} x {n_docs} = {n_terms * n_docs:,} 칸")
print(f"0이 아닌 원소   : {nnz:,} 개")
print(f"희소도(sparsity): {(1 - nnz / (n_terms * n_docs)) * 100:.2f} %  <- 대부분이 0")

행렬 크기       : 846 x 10 = 8,460 칸
0이 아닌 원소   : 1,095 개
희소도(sparsity): 87.06 %  <- 대부분이 0


---
## 문서 수가 고정일 때 TDM 크기를 줄이는 전처리 방법

TDM 의 크기는 **(어휘 수 x 문서 수)** 이므로, 문서 수가 10개로 고정되어 있다면  
**어휘 수**를 줄일 수 밖에 없다.

**1) 형태소 분석 + 품사 필터링 — 효과가 가장 크다**

공백 토큰화는 `데이터를 / 데이터가 / 데이터의` 를 모두 다른 단어로 센다.  
형태소 분석으로 조사를 떼어내는 것만으로 **1,703개 → 846개(50.3% 축소)**.  
조사(J\*), 어미(E\*), 접사(X\*), 기호(S\*)는 의미가 없으므로 품사로 일괄 제거한다.

**2) 원형 복원(표제어 추출)**

`증가했다 / 증가하며 / 증가한` 을 `증가하다` 하나로 합쳐 열을 줄인다.

**3) 불용어 제거**

`것, 등, 기자, 이번` 처럼 어느 기사에나 나오지만 문서를 구분하지 못하는 단어를  
사전으로 제거한다. 열이 줄고 동시에 노이즈도 줄어든다.

**4) 정규화**

대소문자 통일(AI/ai), 숫자 일반화(935건/93건 → `<NUM>`), 한 글자 토큰 제거, 표기 통일(오픈AI/OpenAI).

---

**5) 표현 자체를 바꾸기**

> **메모리 관점** — 희소행렬(scipy sparse)로 저장한다.  
> 이번 TDM 은 **87.06% 가 0**이므로, 밀집 배열로 두면 대부분의 메모리가 낭비된다.  
> 이를 (행, 열, 개수)로 변환하여 관리하면 74.1%의 메모리 절약 효과를 얻을 수 있다.

---
# 2. TF-IDF 계산

- **TF** (Term Frequency) = 문서 내 단어 빈도 / 문서 내 전체 단어 수
- **IDF** (Inverse Document Frequency) = log(전체 문서 수 / 그 단어가 등장한 문서 수)
- **TF-IDF** = TF x IDF

## 2.1 직접 계산

In [ ]:
tf = df_TDM / df_TDM.sum(axis=0)
idf = np.log10(len(df_TDM.T) / (df_TDM!=0).sum(axis=1))
tf_idf = tf.T * idf

print("TF-IDF shape (단어 x 문서) :", tf_idf.shape)
tf_idf.round(3).head()

TF-IDF shape (단어 x 문서) : (10, 846)


단어,분기,인공지능,AI,규모,언어,모델,LLM,발생,보안_취약점,늘어나다,...,인해,승객,영업,운전자,사이에서,혼란,대변인,질문,구체,이유
D00,0.021,0.005,0.035,0.004,0.007,0.002,0.015,0.008,0.005,0.015,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D01,0.000,0.000,0.019,0.000,0.000,0.008,0.000,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D02,0.000,0.006,0.007,0.000,0.000,0.017,0.000,0.002,0.003,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D03,0.000,0.000,0.000,0.007,0.000,0.000,0.000,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
D04,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [117]:
# IDF 값 비교 : 여러 문서에 나올수록 IDF 가 작아진다
df_idf = pd.DataFrame({"단어": vocab,
                       "등장 문서 수(DF)": [np.count_nonzero(df_TDM.T.values[:, j]) for j in range(len(vocab))],
                       "IDF": idf.round(3),
                       "총 빈도": TDM.sum(axis=1)})

print("[여러 문서에 두루 등장 -> IDF 낮음]")
display(df_idf.sort_values(["등장 문서 수(DF)", "총 빈도"], ascending=[False, False]).head(8))
print("\n[한 문서에만 등장 -> IDF 최대]")
display(df_idf[df_idf["등장 문서 수(DF)"] == 1].sort_values("총 빈도", ascending=False).head(5))

[여러 문서에 두루 등장 -> IDF 낮음]


,단어,등장 문서 수(DF),IDF,총 빈도
단어,,,,
모델,모델,5,0.301,24
AI,AI,4,0.398,35
서비스,서비스,4,0.398,13
이용자,이용자,4,0.398,12
기록,기록,4,0.398,11
미국,미국,4,0.398,10
활용,활용,4,0.398,9
기업,기업,4,0.398,8



[한 문서에만 등장 -> IDF 최대]


,단어,등장 문서 수(DF),IDF,총 빈도
단어,,,,
무효,무효,1,1.0,18
특허,특허,1,1.0,16
YMTC,YMTC,1,1.0,14
심판,심판,1,1.0,12
GWH,GWH,1,1.0,12


## 2.2 sklearn TfidfVectorizer 활용

sklearn 은 스무딩된 IDF( `log((1+n)/(1+df)) + 1` )와 L2 정규화를 기본 적용하므로  
값의 절대 크기는 직접 계산과 다르다. 하지만 **상위 단어의 순서는 거의 같다.**

In [118]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(lowercase=False)
tfidf_sk = tfidf_vect.fit_transform(docs_joined)
sk_terms = tfidf_vect.get_feature_names_out()

df_tfidf_sk = pd.DataFrame(tfidf_sk.toarray().T, index=sk_terms, columns=doc_ids)

print("sklearn TF-IDF shape :", tfidf_sk.shape)
df_tfidf_sk.round(4).head(5)

sklearn TF-IDF shape : (10, 844)


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
10,0.0,0.0,0.0000,0.0,0.0426,0.0000,0.0000,0.0000,0.0000,0.0
10대,0.0,0.0,0.0000,0.0,0.0426,0.0000,0.0000,0.0000,0.0000,0.0
12월,0.0,0.0,0.0385,0.0,0.0000,0.0000,0.0000,0.0215,0.0000,0.0
17,0.0,0.0,0.0000,0.0,0.0000,0.0000,0.0000,0.0000,0.1201,0.0
1월,0.0,0.0,0.0000,0.0,0.0000,0.0301,0.0392,0.0000,0.0446,0.0


## 2.3 [비교] 단순 빈도 상위어 vs TF-IDF 상위어

In [119]:
for i, did in enumerate(doc_ids):
    top_tf    = df_TDM[did].sort_values(ascending=False).head(5).index.tolist()
    top_tfidf = tf_idf.T[did].sort_values(ascending=False).head(5).index.tolist()
    print(f"\n{did} [{doc_cats[i]}] {doc_titles[i][:34]}")
    print(f"   빈도   : {top_tf}")
    print(f"   TF-IDF : {top_tfidf}")


D00 [AI/보안] 올 2분기 AI·LLM서 발생한 취약점, 전년 比 14배 폭증
   빈도   : ['AI', '취약점', '서비스', '늘어나다', '보고서']
   TF-IDF : ['취약점', 'AI', '보고서', '카스퍼스키', '빠르다']

D01 [AI] AI '데이터 벽'에 갇힌다...SKT, 사후학습·추론으로 돌
   빈도   : ['데이터', 'AI', '학습', '모델', '활용']
   TF-IDF : ['데이터', '학습', 'AI', '웹', '품질']

D02 [AI] 오픈AI '아스트라', 보안·안전 기능도 막강...7대 기능 
   빈도   : ['모델', '오픈AI', '아스트라', '사이버', '수준']
   TF-IDF : ['오픈AI', '아스트라', '수준', '평가', '능력']

D03 [게임] 라이엇게임즈 공동 창업자 "LoL MMO는 '성배'…개발 지속
   빈도   : ['개발', '라이엇_게임즈', 'MMO', '프로젝트', '메릴']
   TF-IDF : ['라이엇_게임즈', 'MMO', '메릴', '개발', '프로젝트']

D04 [게임] 스마일게이트 신작 '이클립스', MMORPG 공식 비틀었다
   빈도   : ['보스', '이용자', '시스템', '이클립스', '장비']
   TF-IDF : ['보스', '장비', '이클립스', '핵심', '방송']

D05 [자동차/배터리] 中, 1~7월 전기차 배터리 점유율 70% 넘겨
   빈도   : ['GWH', '증가', '점유율', '전기차', '기록']
   TF-IDF : ['GWH', '점유율', '사용량', '증가', '전기차']

D06 [자동차] BMW, 전기차 누적 인도 200만대 돌파…i3 첫 인도 이후
   빈도   : ['BMW', '순수전기차', '그룹', '인도', '판매']
   TF-IDF : ['BMW', '순수전기차', '인도', '그룹', '코리아']

D07 [반도체] 마이크론, YMTC 특허 7번째 무효 판단 받았다.

---
## TF-IDF 가 TDM 보다 유리한 점 / TF 와 IDF 의 의미

**■ TF (Term Frequency, 단어 빈도)**

"이 단어가 **이 문서 안에서** 얼마나 중요한가"를 보는 값이다.

**■ IDF (Inverse Document Frequency, 역문서 빈도)**

"이 단어가 **문서를 구분하는 데** 얼마나 쓸모 있는가"를 코퍼스 전체에서 보는 값이다.  
`log(전체 문서 수 / 그 단어가 등장한 문서 수)` 이므로, 많은 문서에 두루 나오는 단어일수록  
0에 가까워지고 소수 문서에만 나오는 단어일수록 커진다.

> IDF 는 **코퍼스가 스스로 만들어낸 불용어 가중치**라고 볼 수 있다.

**■ TF x IDF 의 의미**

"**이 문서에는 자주 나오지만, 다른 문서에는 잘 안 나오는 단어**"에 높은 점수를 준다.  
이것이 곧 그 문서의 주제어이자 문서를 구분하는 특징이다.

**■ TDM 대비 유리한 점**

1. **주제어가 제대로 드러난다.** D00(AI 취약점 기사)의 단순 빈도 1위는 `AI` 였지만,
   TF-IDF 1위는 `취약점` 이고 `카스퍼스키` 가 상위로 올라온다. `AI` 는 4개 문서에  
   공통으로 나와 변별력이 낮다는 것을 IDF 가 잡아낸 것이다.
2. **문서 길이의 영향을 제거한다.** 빈도 행렬은 긴 문서의 값이 전반적으로 커서  
   길이가 비슷하다는 이유만으로 유사해 보이는 문제가 있다.

---

# 3. N-gram

BoW/TDM 은 단어 순서를 버린다. n-gram 은 인접한 n개 토큰을 하나의 피처로 만들어  
국소적인 어순과 연어(collocation)를 담는다.

## 3.1 N-gram 생성

In [123]:
def make_ngrams(tokens, n):
    """토큰 리스트에서 n개씩 묶어 n-gram 튜플 리스트를 만든다"""
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


sample_tokens = doc_ls[5][:8]
print("원본 토큰 :", sample_tokens)
print("2-gram   :", make_ngrams(sample_tokens, 2)[:4])
print("3-gram   :", make_ngrams(sample_tokens, 3)[:4])

# nltk 로도 동일한 결과를 얻을 수 있다
try:
    from nltk.util import ngrams as nltk_ngrams
    print("\n[nltk.util.ngrams]")
    print("2-gram   :", list(nltk_ngrams(sample_tokens, 2))[:4])
except ImportError:
    print("\nnltk 미설치 -> 직접 구현 결과만 사용")

원본 토큰 : ['1월', '7월', '세계', '전기차', '배터리', '시장', '분석', '결과']
2-gram   : [('1월', '7월'), ('7월', '세계'), ('세계', '전기차'), ('전기차', '배터리')]
3-gram   : [('1월', '7월', '세계'), ('7월', '세계', '전기차'), ('세계', '전기차', '배터리'), ('전기차', '배터리', '시장')]

[nltk.util.ngrams]
2-gram   : [('1월', '7월'), ('7월', '세계'), ('세계', '전기차'), ('전기차', '배터리')]


## 3.2 N-gram TDM 생성

sklearn 의 `ngram_range` 로 1-gram / 2-gram / 3-gram / 1~3-gram 통합 TDM 을 만든다.

In [131]:
ngram_tdms = {}
for label, rng in [("1-gram", (1, 1)), ("2-gram", (2, 2)),
                   ("3-gram", (3, 3)), ("1~3-gram 통합", (1, 3))]:
    cv = CountVectorizer(lowercase=False, ngram_range=rng)
    m = cv.fit_transform(docs_joined)
    ngram_tdms[label] = (m, cv.get_feature_names_out())
    print(f"{label:>14} : {m.shape[1]:>5}개 피처")

# 통합 TDM 을 DataFrame 으로 저장
m_all, terms_all = ngram_tdms["1~3-gram 통합"]
df_TDM_ngram = pd.DataFrame(m_all.toarray().T, index=terms_all, columns=doc_ids)
df_TDM_ngram.index.name = "n-gram"
print("\n1~3-gram 통합 TDM shape :", df_TDM_ngram.shape)
display(df_TDM_ngram)

        1-gram :   844개 피처
        2-gram :  1622개 피처
        3-gram :  1722개 피처
   1~3-gram 통합 :  4188개 피처

1~3-gram 통합 TDM shape : (4188, 10)


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
n-gram,,,,,,,,,,
10,0,0,0,0,1,0,0,0,0,0
10 진영,0,0,0,0,1,0,0,0,0,0
10 진영 하이,0,0,0,0,1,0,0,0,0,0
10대,0,0,0,0,1,0,0,0,0,0
10대 10,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
힘겹다 특허분쟁,0,0,0,0,0,0,0,1,0,0
힘겹다 특허분쟁 YMTC,0,0,0,0,0,0,0,1,0,0
힘입다,0,0,0,0,0,0,1,0,0,0


## 3.4 [실험] n 이 커질 때 무슨 일이 일어나는가

- **1회성(hapax) 비율** : 전체에서 딱 1번만 등장한 피처의 비율
- **2개 이상 문서 공유** : 두 개 이상의 문서에 나타나는 피처의 비율
  (= 문서 비교에 실제로 쓸 수 있는 피처의 비율)

In [ ]:
rows = []
for label, (m, terms) in ngram_tdms.items():
    arr = m.toarray()
    n_feat = arr.shape[1]
    nnz = int(np.count_nonzero(arr))
    hapax = int((arr.sum(axis=0) == 1).sum())             # 전체에서 딱 1번만 등장
    df_cnt = (arr > 0).sum(axis=0)                         # 등장 문서 수
    rows.append({
        "n": label,
        "어휘(피처) 수": n_feat,
        "행렬 크기": f"{n_feat} x {len(doc_ids)}",
        "희소도(%)": round((1 - nnz / (n_feat * len(doc_ids))) * 100, 2),
        "1회성 비율(%)": round(hapax / n_feat * 100, 1),
    })
pd.DataFrame(rows)

,n,어휘(피처) 수,행렬 크기,희소도(%),1회성 비율(%),2개 이상 문서 공유(%)
0,1-gram,844,844 x 10,87.07,60.4,20.7
1,2-gram,1622,1622 x 10,89.93,94.3,0.6
2,3-gram,1722,1722 x 10,90.00,98.4,0.0
3,1~3-gram 통합,4188,4188 x 10,89.38,89.2,4.4


---
## N-gram 의 크기를 선택할 때 고려해야 할 사항

**■ N-gram 을 쓰는 이유**

BoW/TDM 은 단어 순서를 담지 못하고,  
`무효 판단` 처럼 두 단어가 붙어야 의미가 완성되는 표현을 잡지 못한다.

**■ n 이 커질 때의 장점**

- **표현력**이 좋아진다. 고유명사·전문용어·연어를 분석에 활용할 수 있다.
- **문맥이 반영**되어 다의어 구분에 유리하다. (`애플 주가`의 애플 vs `애플 사과`의 애플)

- **부정 표현**을 살릴 수 있다. `좋다` 와 `좋지 않다` 를 감성분석에서 구분할 수 있다.

**■ n 이 커질 때의 단점**

- **피처 수가 폭증한다.** 문서 수는 그대로 10개인데 열만 5배로 늘어난다.
- **희소성이 극심해진다.** '2개 이상 문서가 공유하는 피처' 비율이 낮아지게 되고,  
  **3-gram 은 전부 한 문서에만 존재해서 문서 간 비교에 아무 도움이 되지 않는다.**
   - 데이터 희소 문제로 언어모델의 확률 추정이 불안정해져 **스무딩**(라플라스,  
      백오프, 크네저-네이 등)이 필요해진다.
- **연산량과 메모리**가 n 에 따라 빠르게 증가한다.

**■ 적절한 n의 크기를 선택하는 법**

1. **코퍼스 크기를 확인한다.** 문서가 적으면 n=2 만 되어도  
   대부분의 n-gram 이 1회성이 된다. n 을 키우려면 데이터가 훨씬 많아야 한다.
2. **분석 목적을 본다.**
   - 문서 분류/검색 → `ngram_range=(1,2)` 가 표준적인 선택
   - 고유명사·연어 추출 → 2~3 gram
   - 문자 단위 언어모델·오타 교정 → char n-gram 3~5
3. **n 만 키우지 말고 필터를 함께 건다.** 희귀 n-gram 은 제거한다.  
4. 결론적으로 n 은 **'표현력 vs 희소성'의 트레이드오프**이며,  
   보통 (1,2) 에서 시작해 검증 성능으로 (1,3) 확장 여부를 결정한다.

---
# 4. 단어-문맥 동시등장 행렬 (Co-occurrence Matrix)

중심 단어 기준 좌우 `window_size` 만큼을 '문맥'으로 보고 함께 등장한 횟수를 센다.

## 4.1 동시등장 행렬 생성 함수

In [134]:
def build_cooccurrence(tokens, window_size=2, vocab_list=None):
    """
    tokens      : 한 문서의 토큰 리스트
    window_size : 중심 단어 좌우로 몇 칸까지를 문맥으로 볼지
    return      : (동시등장 행렬 DataFrame, 어휘 리스트)
    """
    if vocab_list is None:
        vocab_list = sorted(set(tokens))
    w2i = {w: i for i, w in enumerate(vocab_list)}

    n = len(vocab_list)
    co = np.zeros((n, n), dtype=int)

    for center_i, center in enumerate(tokens):
        if center not in w2i:
            continue
        start = max(0, center_i - window_size)
        end   = min(len(tokens), center_i + window_size + 1)
        for ctx_i in range(start, end):
            if ctx_i == center_i:            # 자기 자신 위치는 세지 않는다
                continue
            ctx = tokens[ctx_i]
            if ctx not in w2i:
                continue
            co[w2i[center], w2i[ctx]] += 1

    return pd.DataFrame(co, index=vocab_list, columns=vocab_list), vocab_list


# 작은 예제로 동작 확인
demo = "오늘 동물원 에서 원숭이 보다 원숭이 바나나 먹다".split()
demo_co, _ = build_cooccurrence(demo, window_size=1)
print("예제 토큰 :", demo)
demo_co

예제 토큰 : ['오늘', '동물원', '에서', '원숭이', '보다', '원숭이', '바나나', '먹다']


,동물원,먹다,바나나,보다,에서,오늘,원숭이
동물원,0,0,0,0,1,1,0
먹다,0,0,1,0,0,0,0
바나나,0,1,0,0,0,0,1
보다,0,0,0,0,0,0,2
에서,1,0,0,0,0,0,1
오늘,1,0,0,0,0,0,0
원숭이,0,0,1,2,1,0,0


## 4.2 실제 기사에 적용 — D01 (AI '데이터 벽')

In [136]:
TARGET = 1                       # 분석할 문서 인덱스
target_tokens = doc_ls[TARGET]
print(f"대상 문서 : {doc_ids[TARGET]} | {doc_titles[TARGET]}")
print(f"토큰 수 {len(target_tokens)} / 고유 단어 {len(set(target_tokens))}")

WINDOW = 2
co_df, co_vocab = build_cooccurrence(target_tokens, window_size=WINDOW)
print(f"\n동시등장 행렬 shape : {co_df.shape}  (window_size={WINDOW})")

# 전체를 다 찍으면 볼 수 없으므로 최빈 단어 12개만 잘라서 본다
top_words = [w for w, _ in Counter(target_tokens).most_common(12)]
co_df.loc[top_words, top_words]

대상 문서 : D01 | AI '데이터 벽'에 갇힌다...SKT, 사후학습·추론으로 돌파구
토큰 수 277 / 고유 단어 170

동시등장 행렬 shape : (170, 170)  (window_size=2)


,데이터,AI,학습,모델,활용,문제,웹,품질,방식,기업,성능,고갈
데이터,4,3,3,3,3,2,3,3,1,2,0,4
AI,3,0,1,2,1,2,1,0,2,2,1,0
학습,3,1,0,2,0,0,0,0,2,0,0,0
모델,3,2,2,0,0,0,0,0,0,0,2,0
활용,3,1,0,0,0,1,0,0,2,1,0,0
문제,2,2,0,0,1,0,0,1,0,0,0,0
웹,3,1,0,0,0,0,0,0,0,0,0,1
품질,3,0,0,0,0,1,0,0,0,0,0,0
방식,1,2,2,0,2,0,0,0,0,0,0,0
기업,2,2,0,0,1,0,0,0,0,0,0,1


## 4.3 특정 단어와 함께 등장한 단어 (문맥)

In [139]:
for word in ["데이터", "AI", "모델"]:
    if word in co_df.index:
        top_ctx = co_df.loc[word].sort_values(ascending=False).head(6)
        top_ctx = top_ctx[top_ctx > 0]
        print(f"\n'{word}' 의 문맥 단어 (window={WINDOW})")
        print("  " + ", ".join(f"{w}({c})" for w, c in top_ctx.items()))


'데이터' 의 문맥 단어 (window=2)
  고갈(4), 데이터(4), 합성(4), 활용(3), 가치(3), 학습(3)

'AI' 의 문맥 단어 (window=2)
  데이터(3), 기업(2), 문제(2), 방식(2), 모델(2), 연구원(2)

'모델' 의 문맥 단어 (window=2)
  데이터(3), AI(2), 성능(2), 학습(2), 개발(1), 게재(1)


## 4.4 [실험] window_size 에 따른 변화

In [140]:
rows = []
for w in [1, 2, 5, 10]:
    cdf, _ = build_cooccurrence(target_tokens, window_size=w)
    arr = cdf.values
    n = arr.shape[0]
    nnz = int(np.count_nonzero(arr))
    key = "데이터" if "데이터" in cdf.index else cdf.sum().idxmax()
    top5 = cdf.loc[key].sort_values(ascending=False).head(5)
    rows.append({
        "window_size": w,
        "0이 아닌 칸": nnz,
        "밀도(%)": round(nnz / (n * n) * 100, 2),
        "총 카운트 합": int(arr.sum()),
        f"'{key}'의 이웃 top5": ", ".join(top5.index.tolist()),
    })
pd.DataFrame(rows)

,window_size,0이 아닌 칸,밀도(%),총 카운트 합,'데이터'의 이웃 top5
0,1,522,1.81,552,"웹, 합성, 활용, 얼마나, 품질"
1,2,997,3.45,1102,"고갈, 데이터, 합성, 활용, 가치"
2,5,2282,7.90,2740,"데이터, AI, 학습, 웹, 고갈"
3,10,4081,14.12,5430,"데이터, AI, 학습, 웹, 모델"


---
## window_size 설정 시 고려할 점 / 크거나 작을 때의 영향

**■ window_size 가 하는 일**

'문맥'의 범위를 정하는 값이다. 중심 단어의 좌우로 몇 칸까지를 "같이 나온 단어"로  
볼 것인지를 결정하며, 이 값이 곧 **이 행렬이 어떤 종류의 유사도를 학습하는지**를 바꾼다.

**■ window_size 가 작을 때 (1~2)**

*장점*
- 바로 옆에 붙어 있는 단어만 세므로 **구문적(syntactic) 관계**를 잡는다.  
  연어, 복합명사, 고정 표현(`합성 데이터`, `무효 심판`)이 선명하게 드러난다.  
- 행렬이 희소해 노이즈가 적고 계산이 가볍다.

*단점*
- **통계량이 부족하다.** 실험에서 window=1 일 때 0이 아닌 칸은 522개(밀도 1.81%)로  
  window=10(4,081개, 14.12%)의 8분의 1 수준이었다.  
  문서가 짧으면 대부분의 단어쌍이 0이라 학습할 정보 자체가 모자란다.
- 조금만 떨어져 있어도 관계를 놓친다.

**■ window_size 가 클 때 (5~10 이상)**

*장점*
- 문장·문단 수준의 **주제적(semantic/topical) 관계**를 잡는다.  
  실제로 `데이터` 의 이웃 상위 5개가 눈에 띄게 달라졌다.

  | window | '데이터'의 이웃 top5 | 성격 |
  |---|---|---|
  | 1 | 웹, 합성, 활용, 얼마나, 품질 | 바로 붙어 쓰이는 **연어** |
  | 10 | 데이터, AI, 학습, 웹, 모델 | 기사 전체의 **주제어** |

- 동시등장 횟수가 많아져 통계가 안정적이다(총 카운트 합 552 → 5,430).

*단점*
- 행렬 **밀도가 급격히 상승**(1.81% → 14.12%)해 메모리와 연산량이 커진다.
- 관계없는 단어까지 함께 세면서 **노이즈가 늘고**, 결과가 '이 문서에 자주 나오는 단어'와  
  구분되지 않게 된다. 즉 변별력이 떨어진다.
- **문장 경계를 넘어가면** 실제로는 무관한 단어를 이웃으로 세게 된다.  
  그래서 실무에서는 문장 단위로 잘라서 윈도를 적용하는 경우가 많다.

**■ 설정할 때 고려할 사항**

1. **목적이 무엇인가**
   - 구문/연어/복합어 추출, 유의어 찾기 → 작은 윈도(1~3)
   - 주제 유사도, 토픽 성격의 임베딩 → 큰 윈도(5~10)
2. **언어와 토큰 단위** — 한국어는 어순이 비교적 자유롭고 형태소로 쪼개면 토큰 수가
   늘어나므로, 영어와 같은 감각으로 잡으면 실제 포착 범위가 좁아진다.  
   (여기선 이미 조사·어미를 제거하여 window=2 도 원문 기준으로는 4~5 어절에 해당한다.)
3. **데이터 양** — 코퍼스가 작으면 윈도를 키워 통계를 확보하는 편이 낫고,  
   충분히 크면 윈도를 좁혀 정밀도를 올리는 편이 낫다.

**■ 결론**

window_size 는 **'문법적 관계 vs 주제적 관계'의 조절 파라미터**다.
정답은 없고, 만들려는 것이 무엇인지에 따라 정한 뒤 결과를 눈으로 확인해 조정하는 것이 좋다.